# 05 — Crop-row and plant segmentation from UAV RGB images

## Objective

This notebook develops a method to **segment and count plants in RGB UAV images of crop rows**.

The dataset contains near-vertical RGB images of experimental plots. Plants are generally arranged in straight, regularly spaced rows, with some missing plants and occasional overlaps. Bounding-box annotations are available and will be used for validation.

## Previous work

In notebook `04_explore_uav_rgb`, we characterised the UAV RGB dataset and analysed the associated plant annotations.

A subset of **9 representative images** was selected as a toy dataset for developing the segmentation method.

The **Excess Green Index (ExG)** was tested as a vegetation signal because green vegetation tends to have a relatively higher green-channel response than soil and background.

ExG provides a simple way to enhance vegetation in RGB images, but the resulting signal is not sufficient on its own to reliably separate individual plants.

## Approach

The segmentation method developed here exploits both the **vegetation signal** and the **geometric structure of the crop rows**.

The workflow is:

```text
RGB image
    ↓
ExG / vegetation signal
    ↓
Estimate crop-row orientation
    ↓
Rotate image
    ↓
Detect crop-row centres
    ↓
Extract crop-row regions
    ↓
Segment vegetation within each row
    ↓
Analyse vegetation distribution along the row
    ↓
Estimate expected plant positions
    ↓
Detect missing plants
    ↓
Validate against available annotations
```

The method is first developed on a single representative image. Parameters and intermediate results are inspected visually to understand the behaviour of each processing step.

Once the approach has been validated, the resulting operations will be implemented as reusable functions in src/opencropphenotyping/.

## Notebook structure

1. RGB and vegetation signal
2. Annotation-based characterisation of the vegetation signal
3. Crop-row geometry
4. Row-based vegetation segmentation
5. Plant detection and counting
6. Validation against annotations
7. Conclusions and next steps


## 1. RGB and vegetation signal

The RGB images are first inspected to characterise their spatial dimensions, colour channels and pixel-value distributions.

The Excess Green Index (ExG) is then computed as a simple vegetation signal. Its distribution is analysed both on a representative image and across the toy dataset to assess whether it provides sufficient contrast between vegetation and background.

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json

project_root = Path.cwd().parent
dataset_dir = project_root/ "data/raw/toy_datasets/toy_segmentation"

image_paths = list(dataset_dir.glob("*.png"))

print(f"Number of images: {len(image_paths)}")


In [ ]:
# Look at image dataset

for path in image_paths:
    with Image.open(path) as image:
        print(
            f"{path.name}: "
            f"{image.size[0]} × {image.size[1]} px, "
            f"{image.mode}, "
            f"bands={image.getbands()}"
        )

# Display all images

fig, axes = plt.subplots(3, 3, figsize=(15, 10))

for ax, path in zip(axes.flat, image_paths):
    with Image.open(path) as image:
        ax.imshow(image)

    ax.set_title(path.name)
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Look at RGB channels of one image

image = Image.open(image_paths[0])

rgb = np.asarray(image)

print("Shape:", rgb.shape) # height, width, channels
print("Dtype:", rgb.dtype)

# Extract RGB channels

red = rgb[:, :, 0].astype(np.float32)
green = rgb[:, :, 1].astype(np.float32)
blue = rgb[:, :, 2].astype(np.float32)

In [ ]:
# Display the channels

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(red, cmap="Reds", vmin=0, vmax=255)
axes[0].set_title("Red channel")

axes[1].imshow(green, cmap="Greens", vmin=0, vmax=255)
axes[1].set_title("Green channel")

axes[2].imshow(blue, cmap="Blues", vmin=0, vmax=255)
axes[2].set_title("Blue channel")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Display pixel value distribution for each channel

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(red.ravel(), bins=20)
axes[0].set_title("Red pixel values")
axes[0].set_xlabel("Pixel value")
axes[0].set_ylabel("Frequency")

axes[1].hist(green.ravel(), bins=20)
axes[1].set_title("Green pixel values")
axes[1].set_xlabel("Pixel value")
axes[1].set_ylabel("Frequency")

axes[2].hist(blue.ravel(), bins=20)
axes[2].set_title("Blue pixel values")
axes[2].set_xlabel("Pixel value")
axes[2].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# Compute the Excess of Green Index (ExG)

exg = 2 * green - red - blue

print("ExG dtype:", exg.dtype)

print(
    f"ExG range: {exg.min():.1f} – {exg.max():.1f}"
)
print(
    f"ExG mean: {exg.mean():.1f}"
)
print(
    f"ExG median: {np.median(exg):.1f}"
)

In [ ]:
# Display ExG

fig, ax = plt.subplots(figsize=(12,4))

im = ax.imshow(exg, cmap="RdYlGn")

ax.set_title("Excess of Green (ExG)")
ax.axis("off")

fig.colorbar(im, ax=ax, label="ExG")

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8,4))

ax.hist(exg.ravel())
ax.set_title("Excess of Green (ExG) pixel distribution")
ax.set_xlabel("ExG")
ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()


### ExG distribution across the dataset

The ExG distributions are inspected across the toy dataset to assess whether a common vegetation threshold could be used.


In [ ]:
# Loop over all images to describe ExG pixel distribution

exg_stats = []

for image_path in image_paths:
    image = Image.open(image_path)

    rgb = np.asarray(image)

    red = rgb[:, :, 0].astype(np.float32)
    green = rgb[:, :, 1].astype(np.float32)
    blue = rgb[:, :, 2].astype(np.float32)

    exg = 2 * green - red - blue

    exg_stats.append({
        "image": image_path.name,
        "min": exg.min(),
        "max": exg.max(),
        "mean": exg.mean(),
        "median": np.median(exg),
    })

exg_df = pd.DataFrame(exg_stats)
print(exg_df)
exg_df.describe()

In [ ]:
fig, axes = plt.subplots(
    3, 3,
    figsize=(15, 10),
)

for ax, image_path in zip(axes.flat, image_paths):

    image = Image.open(image_path)
    rgb = np.asarray(image)

    red = rgb[:, :, 0].astype(np.float32)
    green = rgb[:, :, 1].astype(np.float32)
    blue = rgb[:, :, 2].astype(np.float32)

    exg = 2 * green - red - blue

    ax.hist(exg.ravel(), bins=50)

    ax.set_title(image_path.name)
    ax.set_xlabel("ExG")
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# Visually compare RGB and ExG images

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].imshow(image)
axes[0].set_title("RGB image")
axes[0].axis("off")

im = axes[1].imshow(exg, cmap="RdYlGn")
axes[1].set_title("Excess Green (ExG)")
axes[1].axis("off")

fig.colorbar(im, ax=axes[1], label="ExG")

plt.tight_layout()
plt.show()

## 2. Annotation-based characterisation of the vegetation signal

The previous section showed that ExG provides a useful vegetation signal in the RGB images.

The plant annotations now provide reference locations for the individual plants. These annotations are used to characterise the ExG signal inside and outside plant bounding boxes.

The objective is to assess whether ExG provides sufficient contrast between vegetation and soil to define a useful vegetation mask.

This analysis is exploratory: the annotations are used here to understand the vegetation signal, not to train the final segmentation method.

In [ ]:
# Load plant annotations

json_paths = list(dataset_dir.glob("*.json"))

annotations = {}

for path in json_paths:
    with open(path, encoding="utf-8") as f:
        annotations[path.name] = json.load(f)

annotation = annotations["1_hermine_2019_1.json"]

print(f"Number of JSON files: {len(annotations)}")
print(annotation.keys())
print(annotation["categories"])
print(annotation["images"])
print(annotation["annotations"])

In [ ]:
# Link each image to its plant annotations

image_annotations = {}

for data in annotations.values():

    for image in data["images"]:
        image_annotations[image["file_name"]] = {
            "image": image,
            "annotations": [
                annotation
                for annotation in data["annotations"]
                if annotation["image_id"] == image["id"]
            ],
        }

for image_path in image_paths:
    info = image_annotations[image_path.name]

    print(
        image_path.name,
        len(info["annotations"])
    )

In [ ]:
# Extract ExG pixels from plant bounding boxes
 
def extract_bbox_pixels(
    exg: np.ndarray,
    image_annotations: list[dict],
) -> np.ndarray:

    pixels = []

    for annotation in image_annotations:

        x, y, width, height = annotation["bbox"]

        x = int(x)
        y = int(y)
        width = int(width)
        height = int(height)

        bbox = exg[
            y:y + height,
            x:x + width
        ]

        pixels.append(bbox.ravel())

    return np.concatenate(pixels)

In [ ]:
# Get annotations for one image first
# Characterise ExG values inside plant bounding boxes

image_path = image_paths[0]

info = image_annotations[image_path.name]

image = Image.open(image_path)

rgb = np.asarray(image)

red = rgb[:, :, 0]
green = rgb[:, :, 1]
blue = rgb[:, :, 2]

exg = 2 * green - red - blue

plant_exg = extract_bbox_pixels(
    exg,
    info["annotations"],
)

print("Number of pixels:", len(plant_exg))
print("Min:", plant_exg.min())
print("Max:", plant_exg.max())
print("Mean:", plant_exg.mean())
print("Median:", np.median(plant_exg))

In [ ]:
# Display distributions between in and out of the boxes

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    plant_exg,
    bins=50,
    alpha=0.7,
    label="Plant bounding boxes",
)

ax.hist(
    exg.ravel(),
    bins=50,
    alpha=0.5,
    label="Whole image",
)

ax.set_xlabel("ExG")
ax.set_ylabel("Frequency")
ax.set_title("ExG distribution")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Visualisation of the image and its annotation

from matplotlib.patches import Rectangle

# Take first image as example

image_path = image_paths[0]
annotations = image_annotations[image_path.name]["annotations"]

# Plot the image with all bounding boxes

image = Image.open(image_path)

fig, ax = plt.subplots(figsize=(16, 6))

ax.imshow(image)
print(annotations)
for annotation in annotations:

    x, y, width, height = annotation["bbox"]

    rectangle = Rectangle(
        (x, y),
        width,
        height,
        fill=False,
        edgecolor="red",
        linewidth=1,
    )

    ax.add_patch(rectangle)

ax.set_title(
    f"{image_path.name} — "
    f"{len(annotations)} plants"
)

ax.axis("off")

plt.show()


In [ ]:
# Describe differences between pixels in and out annotation boxes

info = image_annotations[image_path.name]
mask = np.zeros(exg.shape, dtype=bool)

for annotation in info["annotations"]:

    x, y, width, height = annotation["bbox"]

    x = int(x)
    y = int(y)
    width = int(width)
    height = int(height)

    mask[
        y:y + height,
        x:x + width
    ] = True

exg_inbox = exg[mask]
exg_outbox = exg[~mask]

print("OUT-BOX")
print(f"Mean: {exg_outbox.mean():.2f}")
print(f"Median: {np.median(exg_outbox):.2f}")
print(f"Q25: {np.percentile(exg_outbox, 25):.2f}")
print(f"Q75: {np.percentile(exg_outbox, 75):.2f}")

print("\nIN-BOX")
print(f"Mean: {exg_inbox.mean():.2f}")
print(f"Median: {np.median(exg_inbox):.2f}")
print(f"Q25: {np.percentile(exg_inbox, 25):.2f}")
print(f"Q75: {np.percentile(exg_inbox, 75):.2f}\n")

thresholds = [25, 50, 75, 100, 125, 150, 175, 200]

for threshold in thresholds:

    in_ratio = np.mean(exg_inbox > threshold)
    out_ratio = np.mean(exg_outbox > threshold)

    print(
        f"ExG > {threshold:3d} : "
        f"in-box = {in_ratio:.2%}, "
        f"out-box = {out_ratio:.2%}"
    )

### ExG thresholding

The ExG distributions show that plant bounding boxes generally contain higher ExG values than the surrounding image.

However, the distributions overlap substantially. A global ExG threshold can therefore identify many vegetation pixels, but it also produces background noise and fragmented vegetation regions.

We therefore do not use ExG thresholding alone to identify individual plants.

Instead, the vegetation signal will be combined with the known geometric structure of the crop rows.

##### Effect of the ExG threshold

In [ ]:
# Show segmentation with a given threshold

threshold = 25

vegetation_mask = exg > threshold

fig, ax = plt.subplots(figsize=(16, 5))

ax.imshow(vegetation_mask, cmap="gray")
ax.set_title(f"ExG vegetation mask — threshold = {threshold}")

ax.axis("off")

plt.show()

In [ ]:
# The same with multiple thresholds

for threshold in [10, 25, 50, 75, 100]:
    vegetation_mask = exg > threshold

    fig, ax = plt.subplots(figsize=(16, 5))

    ax.imshow(vegetation_mask, cmap="gray")
    ax.set_title(f"ExG > {threshold}")

    ax.axis("off")
    plt.show()

## 3. Crop-row orientation

The crop rows are approximately straight and parallel within each image.

Before detecting their positions, we first estimate their orientation on a representative image.

The objective is to determine the direction of the crop rows and assess whether they can be approximated as parallel lines.

In [ ]:
image_path = dataset_dir / "103_DSC01167.png"

image = Image.open(image_path)

plt.figure(figsize=(16, 6))
plt.imshow(image)
plt.axis("off")
plt.show()

In [ ]:
# Try to determine row angles
image = np.array(Image.open(image_path))

height, width = image.shape[:2]

# angles = [-30, -25, -20, -15, -10, -5, 0]

# Candidate crop-row orientations
angles = np.arange(-20, -4, 1)

print("Candidate angles:", angles)

fig, axes = plt.subplots(
    len(angles),
    1,
    figsize=(16, 4 * len(angles))
)

for ax, angle in zip(axes, angles):

    ax.imshow(image)

    theta = np.deg2rad(angle)

    # Centre de l'image
    x0 = width / 2
    y0 = height / 2

    # Direction de la ligne
    dx = np.cos(theta)
    dy = np.sin(theta)

    length = max(width, height)

    x1 = x0 - length * dx
    y1 = y0 - length * dy

    x2 = x0 + length * dx
    y2 = y0 + length * dy

    ax.plot(
        [x1, x2],
        [y1, y2],
        linewidth=2,
    )

    ax.set_title(f"Angle = {angle}°")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Refine the decision by rotating the image at different angle
# and find out the best angle

from scipy.ndimage import rotate

rotated_images = {}

for angle in angles:

    rotated = rotate(
        exg,
        angle=angle,
        reshape=True,
        order=1,
    )

    rotated_images[angle] = rotated

# Compute the profil ExG for each rotation angle
profiles = {}

for angle, rotated_exg in rotated_images.items():

    profile = rotated_exg.mean(axis=1)

    profiles[angle] = profile

In [ ]:
# Plot some profiles

selected_angles = [-15, -14, -13, -12, -11, -10]

fig, axes = plt.subplots(
    len(selected_angles),
    1,
    figsize=(16, 10),
)

for ax, angle in zip(axes, selected_angles):

    ax.plot(profiles[angle])

    ax.set_title(f"Vegetation profile after rotation: {angle}°")
    ax.set_xlabel("Image row")
    ax.set_ylabel("Mean ExG")
    ax.grid()

plt.tight_layout()
plt.show()

In [ ]:
# Select the estimated crop-row orientation
best_angle = -11

# Use the rotated ExG image
rotated_exg = rotated_images[best_angle]

# Get the rotated RGB image
rotated_img = rotate(
        image,
        angle=angle,
        reshape=True,
        order=1,
    )

# Compute the mean ExG value for each image row
row_profile = rotated_exg.mean(axis=1)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(16, 8))

# Display the rotated ExG image
ax[0].imshow(rotated_exg, origin="lower", cmap="gray")
ax[0].axis("off")
ax[0].set_title(
    f"Rotated ExG image ({best_angle}° rotation)"
)

# Display the crop-row profile
ax[1].plot(row_profile)
ax[1].set_xlabel("Image row (pixels)")
ax[1].set_ylabel("Mean ExG")
ax[1].set_title(
    f"Crop-row vegetation profile ({best_angle}° rotation)"
)
ax[1].grid()

plt.tight_layout()
plt.show()


## 4. Crop-row detection

Once the image has been rotated so that the crop rows are approximately horizontal, the vegetation signal can be aggregated along the horizontal axis.

This produces a vertical vegetation profile in which crop rows appear as peaks.

The profile is smoothed before detecting local maxima. A minimum peak distance is used to avoid detecting multiple peaks within the same crop row.

In [ ]:
# Display the vegetation profile
fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(row_profile)

ax.set_xlabel("Image row (pixels)")
ax.set_ylabel("Mean ExG")
ax.set_title("Vertical vegetation profile")

ax.grid()

plt.show()

In [ ]:
# Define a simple vegetation mask
vegetation_mask = rotated_exg > 25

# Count vegetation pixels for each image row
row_vegetation_count = vegetation_mask.sum(axis=1)

# Display the vegetation profile
fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(row_vegetation_count)

ax.set_xlabel("Image row (pixels)")
ax.set_ylabel("Vegetation pixel count")
ax.set_title("Vegetation profile along the vertical axis")

ax.grid()

plt.show()

##### Detect crop-row peaks

The vegetation profile shows clear peaks corresponding to the crop rows.

We now detect these peaks automatically. A small amount of smoothing is applied first to reduce local noise, while a minimum distance between peaks is used to avoid detecting multiple peaks within the same crop row.


In [ ]:
# Use scipy functions to smoothe and find peaks

from scipy.signal import savgol_filter
from scipy.signal import find_peaks

# Smooth the vegetation profile
smoothed_profile = savgol_filter(
    row_profile,
    window_length=51,
    polyorder=2,
)

peaks, properties = find_peaks(
    smoothed_profile,
    distance=100,
    prominence=10,
)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(row_profile, alpha=0.4, label="Original profile")
ax.plot(smoothed_profile, label="Smoothed profile")

ax.scatter(
    peaks,
    smoothed_profile[peaks],
    zorder=3,
    label="Detected peaks",
)

ax.set_xlabel("Image row (pixels)")
ax.set_ylabel("Mean ExG")
ax.set_title("Crop-row peak detection")

ax.legend()
ax.grid()

plt.show()

print("Detected peaks:", peaks)
print("Number of detected peaks:", len(peaks))

In [ ]:
# Plot rows on the image for visual check

fig, ax = plt.subplots(figsize=(16, 6))

ax.imshow(rotated_img)

for peak in peaks:
    ax.axhline(
        peak,
        linewidth=2,
    )

ax.set_title("Detected crop rows")
ax.axis("off")

plt.show()

##### Crop-row spacing

Crop rows are detected from the smoothed vegetation profile using local peak detection.

A minimum distance between peaks prevents multiple detections within the same crop row, while peak prominence helps reject weak local maxima caused by background noise.

The detected crop rows are represented by their vertical positions in the image.

We now analyse the distances between consecutive rows to verify the assumption of approximately regular row spacing.


In [ ]:
# Check inter-rows distance

row_distances = np.diff(peaks)

print("Crop-row positions:", peaks)
print("Distances between rows:", row_distances)

print(f"Mean row spacing: {row_distances.mean():.1f} px")
print(f"Standard deviation: {row_distances.std():.1f} px")
print(f"Minimum row spacing: {row_distances.min():.1f} px")
print(f"Maximum row spacing: {row_distances.max():.1f} px")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.bar(
    np.arange(1, len(row_distances) + 1),
    row_distances,
)

ax.set_xlabel("Consecutive row pair")
ax.set_ylabel("Distance (pixels)")
ax.set_title(f"Distance between consecutive crop rows - Standard Deviation = {row_distances.std():.1f} px")

plt.show()

## 5. Crop-row search bands

The detected crop-row positions are used to define a search band around each row.

For neighbouring rows, the boundary between two bands is placed halfway between their detected positions.

For the outermost rows, the boundaries are estimated from the mean inter-row spacing.

These bands provide the spatial regions in which vegetation will be analysed for subsequent plant detection.

In [ ]:
# Compute boundaries halfway between consecutive rows
row_boundaries = (peaks[:-1] + peaks[1:]) / 2

print("Row positions:", peaks)
print("Row boundaries:", row_boundaries)

# Estimate the average row spacing
mean_spacing = np.mean(np.diff(peaks))

# Define the outer boundaries
top_boundary = peaks[0] - mean_spacing / 2
bottom_boundary = peaks[-1] + mean_spacing / 2

# Keep boundaries inside the image
top_boundary = max(0, top_boundary)
bottom_boundary = min(rotated_img.shape[0], bottom_boundary)

print("Top boundary:", top_boundary)
print("Bottom boundary:", bottom_boundary)

In [ ]:
# Combine outer and internal boundaries
boundaries = np.concatenate([
    [top_boundary],
    row_boundaries,
    [bottom_boundary],
])

print("Boundaries:", boundaries)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.imshow(rotated_img)

# Draw detected row positions
for peak in peaks:
    ax.axhline(
        peak,
        linewidth=2,
        linestyle="-",
    )

# Draw row boundaries
for boundary in boundaries:
    ax.axhline(
        boundary,
        linewidth=1,
        linestyle="--",
        color="red",
    )

ax.set_title("Crop rows and search bands")
ax.axis("off")

plt.show()

## 9. Segment vegetation within crop rows

The crop-row geometry provides a separate search region for each row.

Within each row band, the vegetation signal is analysed along the row direction. The objective is to distinguish plant pixels from soil and background before detecting individual plants.

In [ ]:
# Extract the image region corresponding to each crop row
row_images = []

for i in range(len(peaks)):

    y_start = int(boundaries[i])
    y_end = int(boundaries[i + 1])

    row_image = rotated_img[y_start:y_end, :, :]

    row_images.append(row_image)

    print(
        f"Row {i + 1}: "
        f"{y_start}–{y_end} px, "
        f"shape = {row_image.shape}"
    )


In [ ]:
fig, axes = plt.subplots(
    len(row_images),
    1,
    figsize=(16, 12),
)

for i, (ax, row_image) in enumerate(
    zip(axes, row_images),
    start=1,
):

    ax.imshow(row_image)
    ax.set_title(f"Crop row {i}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Compute ExG for each row region
row_exg = []

for row_image in row_images:

    red = row_image[:, :, 0].astype(np.float32)
    green = row_image[:, :, 1].astype(np.float32)
    blue = row_image[:, :, 2].astype(np.float32)

    exg_row = 2 * green - red - blue

    row_exg.append(exg_row)

An empirical ExG threshold is then applied to obtain a binary vegetation mask.

The threshold is initially set to 25 based on the exploratory analysis above. Its suitability is assessed visually on the resulting masks.

In [ ]:
# Apply an ExG threshold within each crop row
exg_threshold = 25

row_masks = [
    exg > exg_threshold
    for exg in row_exg
]

In [ ]:
fig, axes = plt.subplots(
    len(row_masks),
    1,
    figsize=(16, 12),
)

for i, (ax, mask) in enumerate(
    zip(axes, row_masks),
    start=1,
):

    ax.imshow(mask, cmap="gray")
    ax.set_title(
        f"Crop row {i} — ExG > {exg_threshold}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

## 10. Vegetation signal along crop rows

The binary vegetation masks identify vegetation pixels within each crop-row band.

To identify individual planting positions, the two-dimensional vegetation mask is projected along the crop-row direction.

For each position along the row, the number of vegetation pixels across the row width is computed. This produces a one-dimensional vegetation signal that summarizes the amount of vegetation at each position.

In [ ]:
# Compute the vegetation signal along each crop row

row_profiles = []

for mask in row_masks:

    vegetation_profile = mask.sum(axis=0)

    row_profiles.append(vegetation_profile)

In [ ]:
# Visualise row profiles

fig, axes = plt.subplots(
    len(row_profiles),
    1,
    figsize=(16, 10),
)

for i, (ax, profile) in enumerate(
    zip(axes, row_profiles),
    start=1,
):

    ax.plot(profile)

    ax.set_title(
        f"Vegetation signal along crop row {i}"
    )

    ax.set_xlabel("Position along row (pixels)")
    ax.set_ylabel("Vegetation pixels")

    ax.grid()

plt.tight_layout()
plt.show()

In [ ]:
# Check accuracy between row image and its profile
# Example with the first row

fig, axes = plt.subplots(
    2,
    1,
    figsize=(16, 8),
)

# Vegetation mask
axes[0].imshow(
    row_masks[0],
    cmap="gray",
    # Extend the mask to profile boundaries
    extent=[0, row_masks[0].shape[1] - 1, row_masks[0].shape[0], 0],
    aspect="auto",
)

axes[0].set_title("Crop row 1 — vegetation mask")
axes[0].set_xlim(0, row_masks[0].shape[1] - 1)
axes[0].axis("off")

# Vegetation profile
axes[1].plot(row_profiles[0])

axes[1].set_xlabel("Position along row (pixels)")
axes[1].set_ylabel("Vegetation pixels")
axes[1].set_title(
    "Crop row 1 — vegetation signal"
)
axes[1].set_xlim(0, row_masks[0].shape[1] - 1)
axes[1].grid()

plt.tight_layout()
plt.show()

In [ ]:
# Smooth the row profile

smoothed_row_profile = savgol_filter(
    row_profiles[0],
    window_length=31,
    polyorder=2,
)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(
    row_profiles[0],
    alpha=0.4,
    label="Original profile",
)

ax.plot(
    smoothed_row_profile,
    label="Smoothed profile",
)

ax.set_xlabel("Position along row (pixels)")
ax.set_ylabel("Vegetation pixels")
ax.set_title("Vegetation signal along crop row 1")

ax.legend()
ax.grid()

plt.show()

The smoothed vegetation profile is converted into a binary one-dimensional signal.

A second threshold is applied to the profile itself. Unlike the ExG threshold used to create the pixel-level vegetation mask, this threshold represents the minimum number of vegetation pixels expected across a crop-row band at a given position.

Positions above this threshold define vegetation-supporting sections along the row.

In [ ]:
# Define the vegetation segment

vegetation_threshold = 20

vegetation_signal = (
    smoothed_row_profile > vegetation_threshold
)

# Show vegetation segments

fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(
    smoothed_row_profile,
    label="Smoothed vegetation profile",
)

ax.axhline(
    vegetation_threshold,
    linestyle="--",
    label="Vegetation threshold",
)

ax.fill_between(
    range(len(vegetation_signal)),
    0,
    smoothed_row_profile,
    where=vegetation_signal,
    alpha=0.3,
)

ax.set_xlabel("Position along row (pixels)")
ax.set_ylabel("Vegetation pixels")
ax.set_title("Vegetation sections along crop row 1")

ax.legend()
ax.grid()

plt.show()

## 11. Explore vegetation structure within the row

The one-dimensional vegetation signal describes how vegetation is distributed along the crop row.

The next step is to investigate whether this signal contains spatial patterns that could help locate individual plants.

Local minima are therefore explored as potential separators between neighbouring vegetation structures.

This is an exploratory analysis only. Local minima are not assumed to correspond directly to individual plants, since plant size, leaf architecture and missing plants can produce highly variable vegetation profiles.

In [ ]:
# Explore local minima in the vegetation profile

from scipy.signal import find_peaks

valleys, properties = find_peaks(
    -smoothed_row_profile,
    distance=50,
    prominence=5,
)

for valley, prominence in zip(
    valleys,
    properties["prominences"],
):
    print(
        f"Position: {valley}, "
        f"prominence: {prominence:.2f}"
    )

# Display local minima

fig, axes = plt.subplots(
    2,
    1,
    figsize=(16, 8),
)

# Vegetation mask
axes[0].imshow(
    row_masks[0],
    cmap="gray",
    # Extend the mask to profile boundaries
    extent=[0, row_masks[0].shape[1] - 1, row_masks[0].shape[0], 0],
    aspect="auto",
)

axes[0].set_title("Crop row 1 — vegetation mask")
axes[0].set_xlim(0, row_masks[0].shape[1] - 1)
axes[0].axis("off")

# Vegetation profile
axes[1].plot(
    smoothed_row_profile,
    label="Smoothed vegetation profile",
)

axes[1].scatter(
    valleys,
    smoothed_row_profile[valleys],
    color="red",
    zorder=3,
    label="Local minima",
)

axes[1].axhline(
    vegetation_threshold,
    linestyle="--",
    label="Vegetation threshold",
)

axes[1].set_xlabel("Position along row (pixels)")
axes[1].set_ylabel("Vegetation pixels")
axes[1].set_title("Vegetation profile and local minima")

axes[1].legend()
axes[1].set_xlim(0, row_masks[0].shape[1] - 1)
axes[1].grid()

plt.tight_layout()
plt.show()

## 12. Estimate expected plant positions

The number of planted individuals is known for the selected image: 29 plants per crop row.

Rather than relying on the image boundaries directly, the effective beginning and end of the crop row are first estimated from the vegetation signal.

The row extent is then divided into 29 equal planting intervals. The centre of each interval provides an expected plant position.

These positions form a theoretical reference grid that will be used to search for vegetation evidence around each expected plant location.

In [ ]:
# Detect the vegetation-supporting region along the crop row

vegetation_threshold = 20

vegetation_region = (
    smoothed_row_profile > vegetation_threshold
)

vegetation_pixels = np.where(
    vegetation_region
)[0]

x_start = vegetation_pixels[0]
x_end = vegetation_pixels[-1]

print(f"Estimated row start: {x_start} px")
print(f"Estimated row end: {x_end} px")
print(f"Estimated row length: {x_end - x_start} px")

fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(
    smoothed_row_profile,
    label="Smoothed vegetation profile",
)

ax.axvline(
    x_start,
    linestyle="--",
    label="Estimated row start",
)

ax.axvline(
    x_end,
    linestyle="--",
    label="Estimated row end",
)

ax.set_xlabel("Position along row (pixels)")
ax.set_ylabel("Vegetation pixels")
ax.set_title("Estimated crop-row boundaries")

ax.legend()
ax.grid()

plt.show()

In [ ]:
# Number of plants expected along the crop row

n_plants = 29

# Divide the effective row into equal-sized planting segments

segment_edges = np.linspace(
    x_start,
    x_end,
    n_plants + 1,
)

# Use the center of each segment as the expected plant position

plant_positions = (
    segment_edges[:-1] + segment_edges[1:]
) / 2

plant_positions = np.round(
    plant_positions
).astype(int)

print("Expected plant positions:")
print(plant_positions)

print(
    "Expected intra-row spacing:",
    np.diff(plant_positions)
)

print(
    "Mean expected spacing:",
    np.diff(plant_positions).mean()
)

In [ ]:
# Visualize the expected positions

fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(
    smoothed_row_profile,
    label="Smoothed vegetation profile",
)

ax.scatter(
    plant_positions,
    smoothed_row_profile[plant_positions],
    color="red",
    zorder=3,
    label="Expected plant positions",
)

ax.axvline(
    x_start,
    linestyle="--",
    label="Estimated row start",
)

ax.axvline(
    x_end,
    linestyle="--",
    label="Estimated row end",
)

ax.set_xlabel("Position along row (pixels)")
ax.set_ylabel("Vegetation pixels")
ax.set_title("Expected plant positions")

ax.legend()
ax.grid()

plt.show()

## 13. Define search windows around expected plant positions

The expected plant positions provide a regular reference grid along the crop row.

Because the actual plant centres may deviate from these theoretical positions, a search window is defined around each expected position.

The vegetation signal within these windows can then be used to assess the presence and strength of vegetation near each expected planting position.

In [ ]:
expected_spacing = np.diff(plant_positions).mean()

print(f"Expected intra-row spacing: {expected_spacing:.1f} px")

window_half_width = int(expected_spacing / 2)

print(f"Search window half-width: {window_half_width} px")
print(
    f"Search window width: "
    f"{2 * window_half_width} px"
)

In [ ]:
# Create the search windows

search_windows = []

for position in plant_positions:

    x_start_window = max(
        0,
        position - window_half_width,
    )

    x_end_window = min(
        row_masks[0].shape[1] - 1,
        position + window_half_width,
    )

    search_windows.append(
        (
            x_start_window,
            x_end_window,
        )
    )

print("Search windows:")

for i, (start, end) in enumerate(
    search_windows,
    start=1,
):

    print(
        f"Plant {i}: "
        f"{start}–{end} px "
        f"({end - start} px)"
    )

In [ ]:
# Visualize the search windows

fig, ax = plt.subplots(figsize=(16, 5))

ax.imshow(
    row_masks[0],
    cmap="gray",
    extent=[
        0,
        row_masks[0].shape[1] - 1,
        row_masks[0].shape[0],
        0,
    ],
    aspect="auto",
)

for i, (position, (start, end)) in enumerate(
    zip(plant_positions, search_windows),
    start=1,
):

    ax.axvspan(
        start,
        end,
        alpha=0.15,
        color="red"
    )

    ax.axvline(
        position,
        linestyle="--",
        alpha=0.7,
    )

    ax.text(
        position,
        0.95,
        str(i),
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        color="white"
    )

ax.set_xlim(
    x_start,
    x_end,
)

ax.set_xlabel("Position along row (pixels)")
ax.set_title(
    "Search windows around expected plant positions"
)

plt.show()

## 14. Quantify vegetation within each search window

For each expected plant position, the number of vegetation pixels within its search window is calculated.

This provides a quantitative measure of vegetation presence around each expected planting position.

No presence/absence threshold is applied yet. The distribution of vegetation pixels will first be examined to determine whether planted and empty positions can be distinguished.

In [ ]:
# Count vegetation pixels within each search window

vegetation_pixel_counts = []

for start, end in search_windows:

    window_mask = row_masks[0][:, start:end]

    vegetation_count = np.sum(window_mask)

    vegetation_pixel_counts.append(
        vegetation_count
    )

# Display the vegetation pixel count for each expected plant position

for i, count in enumerate(
    vegetation_pixel_counts,
    start=1,
):

    print(
        f"Plant position {i:02d}: "
        f"{count} vegetation pixels"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))

ax.bar(
    range(1, n_plants + 1),
    vegetation_pixel_counts,
)

ax.set_xlabel("Expected plant position")
ax.set_ylabel("Number of vegetation pixels")
ax.set_title(
    "Vegetation pixels within each search window"
)

ax.set_xticks(
    range(1, n_plants + 1)
)

ax.grid(
    axis="y",
    alpha=0.3,
)

plt.tight_layout()
plt.show()

## 15. Visualize vegetation evidence

The vegetation pixel count provides a quantitative measure of vegetation presence around each expected plant position.

The vegetation evidence is visualized together with the crop-row mask to assess whether low vegetation counts correspond to missing plants or simply to smaller plants.

In [ ]:
# Visualize vegetation evidence at each expected plant position

fig, ax = plt.subplots(figsize=(16, 5))

ax.imshow(
    row_masks[0],
    cmap="gray",
    extent=[
        0,
        row_masks[0].shape[1] - 1,
        row_masks[0].shape[0],
        0,
    ],
    aspect="auto",
)

for i, (position, count) in enumerate(
    zip(
        plant_positions,
        vegetation_pixel_counts,
    ),
    start=1,
):

    ax.axvline(
        position,
        linestyle="--",
        alpha=0.5,
    )

    ax.text(
        position,
        0.15,
        f"{i}\n{count}",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=8,
        color="white"
    )

ax.set_xlim(
    x_start,
    x_end,
)

ax.set_xlabel("Position along row (pixels)")
ax.set_title(
    "Vegetation evidence at expected plant positions"
)

plt.show()

## 16. Calculate vegetation centroids

For each search window, the spatial distribution of vegetation pixels is analysed.

The centroid of the vegetation mask provides an estimate of where the detected vegetation is concentrated within the search window.

This information can be compared with the expected plant position.

In [ ]:
# Calculate the vegetation centroid for each search window

vegetation_centroids = []

for position, (start, end) in zip(
    plant_positions,
    search_windows,
):

    window_mask = row_masks[0][:, start:end + 1]

    y_coords, x_coords = np.where(
        window_mask
    )

    if len(x_coords) == 0:

        vegetation_centroids.append(None)

        continue

    # Convert local x coordinates to image coordinates

    x_coords_global = x_coords + start

    centroid_x = x_coords_global.mean()
    centroid_y = y_coords.mean()

    vegetation_centroids.append(
        (centroid_x, centroid_y)
    )

In [ ]:
for i, (position, centroid) in enumerate(
    zip(
        plant_positions,
        vegetation_centroids,
    ),
    start=1,
):

    if centroid is None:

        print(
            f"Plant position {i:02d}: "
            "no vegetation detected"
        )

    else:

        centroid_x, centroid_y = centroid

        # High offset values can reveal missing plant replaced by grass or border vegetation
        offset = centroid_x - position

        print(
            f"Plant position {i:02d}: "
            f"expected x = {position}, "
            f"centroid x = {centroid_x:.1f}, "
            f"offset = {offset:.1f} px"
        )

## 17. Combine plant detection indicators

The information extracted for each expected plant position is combined into a single table.

For each position, we consider:

- the expected plant position;
- the amount of vegetation detected within the search window;
- the vegetation centroid;
- the centroid offset from the expected position;
- the absolute centroid offset.

This table will be used to define the plant presence criterion.

In [ ]:
import pandas as pd

# Build a summary table for all expected plant positions

plant_data = []

for i, (
    position,
    vegetation_count,
    centroid,
) in enumerate(
    zip(
        plant_positions,
        vegetation_pixel_counts,
        vegetation_centroids,
    ),
    start=1,
):

    if centroid is None:

        centroid_x = np.nan
        centroid_y = np.nan
        offset = np.nan
        abs_offset = np.nan

    else:

        centroid_x, centroid_y = centroid

        offset = centroid_x - position
        abs_offset = abs(offset)

    plant_data.append(
        {
            "plant_position": i,
            "expected_x": position,
            "vegetation_pixels": vegetation_count,
            "centroid_x": centroid_x,
            "centroid_y": centroid_y,
            "offset": offset,
            "abs_offset": abs_offset,
        }
    )

plant_df = pd.DataFrame(plant_data)

plant_df.sort_values(
    ["vegetation_pixels", "abs_offset"]
)

## 18. Define the missing-plant criterion

The expected planting positions provide a regular reference grid along each crop row.

For each position, two complementary indicators are used:

- the amount of vegetation detected within the search window;
- the distance between the expected plant position and the vegetation centroid.

A missing-plant candidate is expected to show weak vegetation
evidence near the theoretical planting position and, in some cases,
a displacement of the vegetation centroid toward neighbouring
vegetation.

These two indicators are therefore used as complementary evidence
rather than as independent proof of plant absence.

The thresholds used to classify potential missing plants are defined
from the observed distributions rather than arbitrarily fixed.

In [ ]:
# Inspect vegetation evidence across all expected plant positions

print("Vegetation pixels:")
print(
    f"Minimum: {plant_df['vegetation_pixels'].min()}"
)
print(
    f"Q1: {plant_df['vegetation_pixels'].quantile(0.25):.0f}"
)
print(
    f"Median: {plant_df['vegetation_pixels'].median():.0f}"
)
print(
    f"Q3: {plant_df['vegetation_pixels'].quantile(0.75):.0f}"
)
print(
    f"Maximum: {plant_df['vegetation_pixels'].max()}"
)

print()

print("Absolute centroid offset:")
print(
    f"Minimum: {plant_df['abs_offset'].min():.1f}"
)
print(
    f"Q1: {plant_df['abs_offset'].quantile(0.25):.1f}"
)
print(
    f"Median: {plant_df['abs_offset'].median():.1f}"
)
print(
    f"Q3: {plant_df['abs_offset'].quantile(0.75):.1f}"
)
print(
    f"Maximum: {plant_df['abs_offset'].max():.1f}"
)

In [ ]:
fig, axes = plt.subplots(
    2,
    1,
    figsize=(16, 8),
)

axes[0].bar(
    plant_df["plant_position"],
    plant_df["vegetation_pixels"],
)

axes[0].set_xlabel("Expected plant position")
axes[0].set_ylabel("Vegetation pixels")
axes[0].set_title(
    "Vegetation evidence by expected plant position"
)
axes[0].set_xticks(
    range(1, n_plants + 1)
)
axes[0].grid(
    axis="y",
    alpha=0.3,
)

axes[1].bar(
    plant_df["plant_position"],
    plant_df["abs_offset"],
)

axes[1].set_xlabel("Expected plant position")
axes[1].set_ylabel("Absolute centroid offset (px)")
axes[1].set_title(
    "Centroid displacement by expected plant position"
)
axes[1].set_xticks(
    range(1, n_plants + 1)
)
axes[1].grid(
    axis="y",
    alpha=0.3,
)

plt.tight_layout()
plt.show()

## 19. Detect missing-plant candidates

To identify potential missing plants, robust thresholds are derived
from the distributions of vegetation evidence and centroid displacement.

The interquartile range (IQR) is used to identify unusually low
vegetation evidence and unusually large centroid displacement.

A position is classified as a missing-plant candidate only when both
conditions are met.

In [ ]:
# Define robust thresholds from the observed distributions

vegetation_q1 = plant_df["vegetation_pixels"].quantile(0.25)
vegetation_q3 = plant_df["vegetation_pixels"].quantile(0.75)
vegetation_iqr = vegetation_q3 - vegetation_q1

offset_q1 = plant_df["abs_offset"].quantile(0.25)
offset_q3 = plant_df["abs_offset"].quantile(0.75)
offset_iqr = offset_q3 - offset_q1

low_vegetation_threshold = (
    vegetation_q1 - 1.5 * vegetation_iqr
)

high_offset_threshold = (
    offset_q3 + 1.5 * offset_iqr
)

print(
    f"Low vegetation threshold: "
    f"{low_vegetation_threshold:.1f}"
)

print(
    f"High offset threshold: "
    f"{high_offset_threshold:.1f}"
)

In [ ]:
plant_df["missing_candidate"] = (
    (
        plant_df["vegetation_pixels"]
        < low_vegetation_threshold
    )
    &
    (
        plant_df["abs_offset"]
        > high_offset_threshold
    )
)

plant_df[
    [
        "plant_position",
        "vegetation_pixels",
        "abs_offset",
        "missing_candidate",
    ]
]

## 20. Missing plant detection

The expected planting positions are compared with the vegetation detected around each position.

A position is considered a missing-plant candidate when vegetation evidence is substantially lower than for the other observed plants and the vegetation centroid is significantly displaced from the expected planting position.

For the current image, this approach identifies plant position 19 as a missing plant.

In [ ]:
# Display missing-plant candidates

missing_candidates = plant_df[
    plant_df["missing_candidate"]
]

print(
    "Missing-plant candidates:"
)

for position in missing_candidates["plant_position"]:
    print(f"Plant position {position}")

## 21. Final missing-plant candidate detection

The complete detection workflow is visualized on the selected crop row.

Expected planting positions are displayed along the row, with the detected missing plant highlighted separately.

This provides a final visual validation of the approach on the current image.

In [ ]:
# Visualize the final plant detection result

fig, ax = plt.subplots(figsize=(18, 5))

# Display the vegetation mask
ax.imshow(
    row_masks[0],
    cmap="gray",
    extent=[
        0,
        row_masks[0].shape[1] - 1,
        row_masks[0].shape[0],
        0,
    ],
    aspect="auto",
)

# Separate detected missing positions from occupied positions
missing_positions = plant_df.loc[
    plant_df["missing_candidate"],
    "plant_position",
].tolist()

for _, row in plant_df.iterrows():

    plant_number = int(row["plant_position"])
    x = row["expected_x"]

    if plant_number in missing_positions:

        ax.scatter(
            x,
            row_masks[0].shape[0] * 0.5,
            marker="x",
            s=150,
            linewidths=3,
            label="Missing plant"
            if plant_number == missing_positions[0]
            else None,
        )

        ax.text(
            x,
            -10,
            f"{plant_number}\nmissing",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    else:

        ax.scatter(
            x,
            row_masks[0].shape[0] * 0.5,
            marker="o",
            s=50,
            facecolors="none",
            edgecolors="red",
            linewidths=1.5,
            label="Expected plant"
            if plant_number == 1
            else None,
        )

        ax.text(
            x,
            -10,
            str(plant_number),
            ha="center",
            va="bottom",
            fontsize=8,
        )

ax.set_xlim(
    plant_df["expected_x"].min() - 50,
    plant_df["expected_x"].max() + 50,
)

ax.set_ylim(
    row_masks[0].shape[0],
    -40,
)

ax.set_xlabel("Position along row (pixels)")
ax.set_ylabel("Image row (pixels)")

ax.set_title(
    "Final plant detection result"
)

ax.legend()

ax.grid(
    alpha=0.2,
)

plt.tight_layout()
plt.show()

## 22. Conclusion

This notebook demonstrated a prototype workflow for detecting
potentially missing crop plants along a single crop row from an RGB
UAV image.

The workflow successfully:

- detected and corrected the crop-row orientation;
- identified crop-row positions and boundaries;
- segmented vegetation using the ExG index;
- estimated regular planting positions along the row;
- quantified vegetation around each expected plant position;
- estimated vegetation centroids within local search windows;
- combined vegetation quantity and centroid displacement as complementary
  indicators of potential missing plants.

For the selected image, the approach identifies plant position 19 as a
missing-plant candidate.

This result provides a first validation of the approach on a single
image. However, the detection criteria still need to be validated on
multiple images and crop rows containing known missing plants and
occupied positions.

The next step is to turn the validated workflow into reusable
functions and evaluate its robustness across different field
conditions, plant sizes, vegetation densities and missing-plant
configurations.